In [26]:
import numpy as np
import pandas as pd
import tensorflow as tf
import pickle
from tensorflow.keras.models import load_model

In [41]:
# load trained model, scaller pickel and onehot encoded
model = load_model('model3.h5')

with open('encoder.pkl', 'rb') as f:
    encoder = pickle.load(f)
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)
with open('gender_encoder.pkl', 'rb') as f:
    gender_encoder = pickle.load(f)

In [68]:
# Example input data
input_data = {
    'CreditScore': 119,
    'Geography': 'France',
    'Gender': 'Female',
    'Age': 42,
    'Tenure': 2,
    'Balance': 0,
    'NumOfProducts': 0,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 101348.88
}

In [69]:
geo_enc=encoder.transform([[input_data['Geography']]])
geo_enc_df=pd.DataFrame(geo_enc, columns=encoder.get_feature_names_out(['Geography']))
geo_enc_df

c:\Users\ankit\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [70]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,119,France,Female,42,2,0,0,1,1,101348.88


In [71]:
# Drop original Geography column
input_df = input_df.drop('Geography', axis=1)

# Add encoded columns
input_df = pd.concat([input_df, geo_enc_df], axis=1)

input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,119,Female,42,2,0,0,1,1,101348.88,1.0,0.0,0.0


In [72]:
if input_df['Gender'].iloc[0] == 'Male':
    input_df['Gender'] = 1                      
else:
    input_df['Gender'] = 0

In [73]:
input_df


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,119,0,42,2,0,0,1,1,101348.88,1.0,0.0,0.0


In [74]:
# scaling the input data
num_col=['CreditScore', 'Age', 'Balance', 'Tenure',  'NumOfProducts', 'EstimatedSalary']
input_df[num_col] = scaler.transform(input_df[num_col])

In [75]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,-5.505232,0,0.290073,-1.041433,-1.226059,-2.629343,1,1,0.028223,1.0,0.0,0.0


In [76]:
pred=model.predict(input_df)
print("probability of churn:",pred[0][0])
if pred[0][0] > 0.5:
    print("Customer is likely to churn.")  
else:
    print("Customer is not likely to churn.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
probability of churn: 0.9840939
Customer is likely to churn.
